In [13]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore", category=UserWarning)
xr.set_options(keep_attrs=True)
dask.config.set(scheduler="threads", num_workers=os.cpu_count())

# ---------------- PATHS ----------------
PET_PATH = r"C:\Drought\Regridding and data clipping\GLEAM\GLEAM_Ep_India_0p05deg_2000-2023.nc"
OUT_DIR  = r"C:\Drought\EDDI"

# -------------- SETTINGS ---------------
BASELINE = slice("2001-01-01", "2020-12-31")   # intersected per k
SCALES   = [1, 3, 6, 9, 12]
CHUNKS   = {"time": 36, "lat": 200, "lon": 200}

# -------------- HELPERS ----------------
def norm_coords(da: xr.DataArray) -> xr.DataArray:
    ren = {}
    if "latitude" in da.dims:  ren["latitude"]  = "lat"
    if "longitude" in da.dims: ren["longitude"] = "lon"
    if "y" in da.dims:         ren["y"] = "lat"
    if "x" in da.dims:         ren["x"] = "lon"
    da = da.rename(ren)
    if da.lat.size > 1 and float(da.lat[1] - da.lat[0]) < 0: da = da.sortby("lat")
    if da.lon.size > 1 and float(da.lon[1] - da.lon[0]) < 0: da = da.sortby("lon")
    return da

def ensure_month_end(time_like) -> pd.DatetimeIndex:
    t = pd.to_datetime(np.asarray(getattr(time_like, "values", time_like)))
    return pd.PeriodIndex(t, freq="M").to_timestamp("M")

def to_monthly_total(da: xr.DataArray) -> xr.DataArray:
    """Return monthly totals (mm/month) and month-end timestamps."""
    units = (da.attrs.get("units", "") or "").lower().replace(" ", "").replace(".", "")
    time = pd.to_datetime(da.time.values)

    # sub-monthly?
    submonthly = False
    if da.time.size > 1:
        diffs = np.diff(time.values).astype("timedelta64[D]").astype(int)
        submonthly = diffs.size > 0 and np.nanmin(diffs) < 27
    if submonthly:
        out = da.resample(time="MS").sum(keep_attrs=True)
        out = out.assign_coords(time=ensure_month_end(out.time))
        out.attrs["units"] = "mm month-1"
        out.attrs["cell_methods"] = "time: sum (monthly)"
        return out

    # monthly mean (mm/day) → totals
    if ("mm/day" in units) or ("mmd-1" in units) or ("mmd−1" in units):
        days = xr.DataArray(pd.DatetimeIndex(time).to_period("M").days_in_month.values,
                            dims=["time"], coords={"time": da.time})
        out = (da * days.astype(da.dtype))
        out = out.assign_coords(time=ensure_month_end(out.time))
        out.attrs.update(da.attrs)
        out.attrs["units"] = "mm month-1"
        out.attrs["cell_methods"] = "time: sum (from mm/day × days_in_month)"
        return out

    # already monthly totals
    out = da.assign_coords(time=ensure_month_end(da.time))
    out.attrs.setdefault("units", "mm month-1")
    out.attrs.setdefault("cell_methods", "time: sum (monthly)")
    return out

def eddi_from_pet_monthly(PETm: xr.DataArray, baseline: slice) -> dict:
    """
    EDDI at k=1/3/6/9/12 as Z-score per calendar month.
    For each k, intersect baseline with the k-month series (Sk). If empty, fallback to full Sk.
    Avoid dropna (coastal NaNs).
    """
    out = {}
    for k in SCALES:
        # Rolling sum without dropna → just drop first k-1 months
        Sroll = PETm.rolling(time=k, min_periods=k).sum()
        Sk = Sroll.isel(time=slice(k-1, None))

        # Per-k baseline intersection
        t_sk = pd.to_datetime(np.asarray(Sk.time.values))
        tmin_k, tmax_k = t_sk.min(), t_sk.max()
        bstart = max(tmin_k, pd.to_datetime(baseline.start))
        bend   = min(tmax_k, pd.to_datetime(baseline.stop))
        base_k = Sk.sel(time=slice(bstart, bend))
        if base_k.time.size == 0:
            base_k = Sk
            bstart, bend = tmin_k, tmax_k
            print(f"[WARN] EDDI k={k}: no overlap with 2001–2020 after rolling; "
                  f"using available {bstart.date()}–{bend.date()} as baseline.")

        # Explicit month coords (robust with dask)
        base_k = base_k.assign_coords(month=("time", pd.to_datetime(base_k.time.values).month))
        mu = base_k.groupby("month").mean("time")
        sd = base_k.groupby("month").std("time")
        sd = xr.where(sd > 1e-6, sd, np.nan)

        Sk = Sk.assign_coords(month=("time", pd.to_datetime(Sk.time.values).month))
        Z = (Sk - mu.sel(month=Sk["month"])) / sd.sel(month=Sk["month"])

        Z = Z.drop_vars("month").astype("float32").assign_coords(time=ensure_month_end(Z.time))
        Z.name = f"EDDI{k}"
        Z.attrs.update({
            "long_name": f"EDDI (k={k} months, PET Z-score vs baseline per calendar month)",
            "standard_name": "eddi",
            "units": "1",
            "baseline_start": str(pd.to_datetime(bstart).date()),
            "baseline_end":   str(pd.to_datetime(bend).date()),
            "source": "GLEAM Ep (PET)"
        })
        out[Z.name] = Z
    return out

# --------------- MAIN -----------------
def main():
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    print("Opening PET…")
    ds = xr.open_dataset(PET_PATH, chunks=CHUNKS)

    # pick PET var
    vE = None
    for v in ds.data_vars:
        if any(tok in v.lower() for tok in ["ep", "epot", "pet", "potential"]):
            vE = v; break
    if vE is None:
        vE = list(ds.data_vars)[0]
        print(f"[INFO] Using first variable in file: {vE}")

    Ep = norm_coords(ds[vE]).chunk(CHUNKS)

    print("Converting to monthly totals (mm/month)…")
    PETm = to_monthly_total(Ep)
    PETm.attrs["units"] = "mm month-1"
    print("PETm time range:", str(pd.to_datetime(PETm.time.values[0]))[:10], "→",
          str(pd.to_datetime(PETm.time.values[-1]))[:10])

    print("Computing EDDI (1/3/6/9/12)…")
    with ProgressBar():
        eddi = eddi_from_pet_monthly(PETm, BASELINE)

    # Align common time across scales
    times = [pd.to_datetime(np.asarray(da.time.values)) for da in eddi.values()]
    tmin = max(t.min() for t in times)
    tmax = min(t.max() for t in times)

    ds_out = xr.Dataset({k: v.sel(time=slice(tmin, tmax)) for k, v in eddi.items()})
    ds_out = ds_out.assign_coords(time=ensure_month_end(ds_out.time)).chunk(CHUNKS)
    ds_out.attrs.update({
        "title": "EDDI (1/3/6/9/12) — India @0.05°",
        "method": "Z-score of PET rolling sums per calendar month (per-k baseline intersected with 2001–2020)",
        "source": "GLEAM Ep (PET)",
    })

    # Single, compressed write (no slab appends)
    enc = {
        v: {"zlib": True, "complevel": 4, "dtype": "float32",
            "chunksizes": (CHUNKS["time"], CHUNKS["lat"], CHUNKS["lon"]),
            "_FillValue": np.float32(np.nan)}
        for v in ds_out.data_vars
    }
    fname = f"EDDI_005deg_{pd.to_datetime(tmin).strftime('%Y%m')}-{pd.to_datetime(tmax).strftime('%Y%m')}.nc"
    out_nc = Path(OUT_DIR) / fname
    print(f"Writing: {out_nc}")
    ds_out.to_netcdf(out_nc, engine="netcdf4", encoding=enc)
    print("Done.")

if __name__ == "__main__":
    main()


Opening PET…
Converting to monthly totals (mm/month)…
PETm time range: 2000-01-31 → 2023-12-31
Computing EDDI (1/3/6/9/12)…
Writing: C:\Drought\EDDI\EDDI_005deg_200012-202312.nc
Done.
